# World Cup Tracker Demos — `trackers` + RF-DETR on SoccerNet

Two shareable football-analytics demos that promote the Roboflow [`trackers`](https://github.com/roboflow/trackers) library, built on top of [RF-DETR](https://github.com/roboflow/rf-detr) and the [`roboflow/sports`](https://github.com/roboflow/sports) building blocks.

1. **Pass Alternatives** — freeze the frame when a player has the ball and overlay the 3 best passing lanes (scored by openness / forward-progress / receiver-space).
2. **Player Speed & Distance** — per-player speed (km/h) and total distance covered, with an end-of-clip leaderboard.

We run on **SoccerNet** (no FIFA broadcast footage), which ships ground-truth tracks with role/team/jersey labels — so the demos run end-to-end **on the GT path (default)** with only a pitch homography for metric numbers, then optionally swap in **RF-DETR + `ByteTrackTracker`** for the "works on any video" story.

The renders use a shared **Roboflow Football-AI visual style** (`common/visual.py`): team-colored `sv.EllipseAnnotator` feet, rounded `sv.LabelAnnotator` chips, a gold ball `sv.TriangleAnnotator`, a bottom-center radar minimap, and a Roboflow-purple HUD with a `powered by trackers` tag.

> Run this notebook from the **repo root** so `world_cup_projects` imports as a package.

In [ ]:
# --- Setup -----------------------------------------------------------------
# On Colab, uncomment to install deps and clone the repos:
# !pip install -q supervision opencv-python pandas
# !pip install -q git+https://github.com/roboflow/trackers.git
# !pip install -q rfdetr ultralytics gdown        # v2 only (detection + pitch model)
#
# For v2 RF-DETR weights without a writable home dir, set RF_HOME to a local folder:
import os
import sys
from pathlib import Path

import world_cup_projects
from world_cup_projects import DEFAULT_ASSETS_DIR
from world_cup_projects.common.soccernet import DEFAULT_TRACKING_ROOT

PKG_ROOT = Path(world_cup_projects.__file__).resolve().parent
REPO_ROOT = PKG_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("RF_HOME", str(PKG_ROOT / "weights" / "rfdetr"))

import cv2
import numpy as np
import matplotlib.pyplot as plt

DATA_ROOT = DEFAULT_TRACKING_ROOT
ASSETS = DEFAULT_ASSETS_DIR
ASSETS.mkdir(parents=True, exist_ok=True)

def show(img_bgr, figsize=(14, 8), title=None):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

print("repo root:", REPO_ROOT)

## 1. Load SoccerNet and auto-pick the best clip

Every SoccerNet game-state clip is 30 s, 1920×1080, 25 fps. We rank all clips by **possession density** (ball glued to a player's feet), player count, and whether both teams are present, then auto-pick the best one. `SNMOT-194` wins.

In [ ]:
from world_cup_projects.common.soccernet import find_sequences, load_sequence
from world_cup_projects.common.clips import rank_clips

seq_dirs = find_sequences(DATA_ROOT, split="test")
ranking = rank_clips(seq_dirs)
for i, clip in enumerate(ranking[:5], 1):
    print(f"{i}. {clip.name}  score={clip.score:7.1f}  | {clip.reason}")

sequence = load_sequence(next(p for p in seq_dirs if p.name == ranking[0].name))
print("\nAuto-picked:", sequence.name, "| frames:", sequence.length, "| fps:", sequence.frame_rate)
show(cv2.imread(str(sequence.frame_path(182))), title=f"{sequence.name} — frame 182")

## 2. Demo — Pass Alternatives (v1, ground-truth tracks)

We find frames with clear possession and ≥3 pass options, score each teammate lane, and pick the most compelling freeze moments. Each lane score combines:

- **openness** — distance of the nearest opponent to the pass line (interception risk),
- **forward progress** — gain toward the attacking direction,
- **receiver space** — room around the receiver.

The cell below previews one freeze still inline, then renders the full MP4 (4 freeze events).

In [ ]:
from world_cup_projects.common.soccernet import iter_gt_detections
from world_cup_projects.pass_alternatives.render import plan_events, render_demo, _draw_pass_overlay

# Pick freeze moments and preview the best one inline.
events = plan_events(sequence, max_events=4)
ev = events[0]
dets = next(d for _, d in iter_gt_detections(sequence, start=ev.frame_idx, end=ev.frame_idx))
frame = cv2.imread(str(sequence.frame_path(ev.frame_idx)))
print(f"freeze @ frame {ev.frame_idx} | carrier team {ev.carrier.team} | option scores {[round(o.score,2) for o in ev.options]}")
show(_draw_pass_overlay(frame, dets, ev), title=f"Pass alternatives — frame {ev.frame_idx}")

In [ ]:
# Render the full pass-alternatives MP4 (plays the clip, freezes on each event).
pass_out = ASSETS / f"pass_alternatives_v1_{sequence.name}.mp4"
manifest = render_demo(sequence, str(pass_out), max_events=4)
manifest

# For the "from raw pixels" v2 (RF-DETR players + ByteTrack), pass a detection source:
#   from functools import partial
#   from world_cup_projects.common.detect import RFDETRDetector, fit_team_classifier, iter_rfdetr_detections
#   det = RFDETRDetector("nano", device="cpu"); clf = fit_team_classifier(sequence, det)
#   src = partial(iter_rfdetr_detections, detector=det, team_classifier=clf)
#   render_demo(sequence, ".../pass_v2.mp4", detections_source=src, version_tag="v2")

## 3. Demo — Player Speed & Distance (v1, height-calibrated)

We accumulate each track's feet position per frame, then convert pixel motion to meters. The default **height calibration** uses each player's own bounding-box height (~1.8 m) as a local meters-per-pixel scale, which adapts to perspective automatically — no pitch model required. Speeds are median-smoothed and implausible jumps (ID switches) are clamped.

In [ ]:
from world_cup_projects.player_stats.speed_distance import collect_tracks, compute_kinematics, format_speed_kmh
from world_cup_projects.player_stats.render import render_demo as render_speed, _draw_speed_labels

# Build tracks once, compute kinematics (height calibration).
frames = list(iter_gt_detections(sequence))
tracks = collect_tracks(iter(frames))
compute_kinematics(tracks, sequence.frame_rate, mode="height")

# Leaderboard
top = sorted(tracks.values(), key=lambda t: t.distance_m, reverse=True)[:5]
for t in top:
    print(f"#{t.track_id:<3d} team {t.team:>2d}  distance {t.distance_m:6.1f} m  peak {format_speed_kmh(t.top_speed_ms)}")

# Preview speed labels on one frame
fi, dets = frames[200]
prev = _draw_speed_labels(cv2.imread(str(sequence.frame_path(fi))), dets, tracks, fi)
show(prev, title=f"Speed labels — frame {fi}")

In [ ]:
# Render the full speed/distance MP4 with the end-of-clip leaderboard.
speed_out = ASSETS / f"player_stats_gt_height_{sequence.name}.mp4"
manifest = render_speed(
    sequence,
    iter(frames),
    tracks,
    str(speed_out),
    frame_loader=lambda fi: cv2.imread(str(sequence.frame_path(fi))),
    calibration="bbox-height (~1.8 m)",
    show_radar=False,        # radar needs the pitch-homography model (see section 5)
)
manifest

## 4. v2 — RF-DETR detection + `trackers.ByteTrackTracker`

The "from raw pixels" pipeline: RF-DETR detects players, `ByteTrackTracker` keeps stable IDs, and a lightweight jersey-color KMeans assigns teams (a fast stand-in for the Siglip `TeamClassifier` in `common/teams.py`). The same pass / speed logic runs unchanged on top.

> RF-DETR auto-selects MPS on macOS; pass `device="cpu"` on older OSes. Weights cache to `RF_HOME` (set in the setup cell).

In [ ]:
# RF-DETR + ByteTrack on a few frames (CPU is slow; use a small range here).
from world_cup_projects.common.detect import RFDETRDetector, fit_team_classifier, iter_rfdetr_detections
import supervision as sv

detector = RFDETRDetector("nano", device="cpu", threshold=0.4)
team_clf = fit_team_classifier(sequence, detector, sample_stride=60, max_frames=300)

fi, dets = next(iter_rfdetr_detections(sequence, detector, team_clf, start=182, end=182))
img = cv2.imread(str(sequence.frame_path(fi)))
palette = sv.ColorPalette.from_hex(["#00BFFF", "#FF1493", "#FFD700"])
players = dets[dets.class_id == 0]
players.class_id = np.where(np.isin(players.data["team"], (0, 1)), players.data["team"], 2).astype(int)
annot = sv.EllipseAnnotator(color=palette, color_lookup=sv.ColorLookup.CLASS, thickness=2)
show(annot.annotate(img.copy(), players), title=f"RF-DETR + ByteTrack — {len(players)} players, team-colored")

## 5. Metric homography + radar minimap (pitch-keypoint model)

For **true** pitch meters and m/s (camera-angle independent) and a top-down radar, we estimate a per-frame homography from Inference `football-field-detection-f07vi/15` (`common/pitch.py`; needs `ROBOFLOW_API_KEY`). This replaces the legacy local `football-pitch-detection.pt` from `roboflow/sports`. When homography is available, pass `mode="homography"` to `compute_kinematics` and `show_radar=True` to the renderer; the demos otherwise **gracefully fall back to height calibration**.

> **Weights note:** the pitch-keypoint model is hosted on Google Drive. In sandboxed/CI environments Google Drive may be blocked (HTTP 403), so the auto-download can fail there — run `ensure_pitch_model()` on a machine with open network access, or drop the `.pt` into `world_cup_projects/.cache/models/`.

In [ ]:
# Try to enable the metric homography path; fall back cleanly if weights are unavailable.
from world_cup_projects.common.pitch import ensure_pitch_model, load_pitch_model, draw_pitch, PITCH_CONFIG

try:
    ensure_pitch_model()
    pitch = load_pitch_model(device="cpu")
    t = pitch(cv2.imread(str(sequence.frame_path(182))))
    print("Pitch homography available:", t is not None)
    show(draw_pitch(PITCH_CONFIG), figsize=(10, 6), title="Radar pitch template")
except Exception as exc:
    print("Pitch model unavailable (expected in blocked/sandboxed envs):")
    print("  ", type(exc).__name__, str(exc)[:160])
    print("\nMetric m/s + radar will activate automatically once the .pt is present.")
    print("Until then the demos use the bbox-height calibration shown above.")

## Notes & credits

- **Tiers:** v1 runs on SoccerNet ground-truth tracks with **no weights**; v2 swaps in **RF-DETR + `trackers.ByteTrackTracker`** and optional pitch homography.
- **Building blocks:** [`trackers`](https://github.com/roboflow/trackers), [RF-DETR](https://github.com/roboflow/rf-detr), and vendored utilities from [`roboflow/sports`](https://github.com/roboflow/sports) (`ViewTransformer`, `SoccerPitchConfiguration`, pitch annotators, `TeamClassifier`) — all Apache-2.0.
- **Data:** SoccerNet game-state tracking clips (no FIFA broadcast footage).
- **CLI equivalents** (full-length renders):
  ```bash
  PYTHONPATH=. python -m world_cup_projects.pass_alternatives.run --sequence SNMOT-194
  PYTHONPATH=. python -m world_cup_projects.player_stats.run --sequence SNMOT-194 --mode height
  ```